## 测试节点
* 定义AgentState
* 

In [ ]:
from typing import Dict, List, Any, Callable
import json
from langchain_core.language_models import BaseLanguageModel
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

class AgentState(BaseModel):
    # 用户问题
    query: str = ""

    # 历史消息 分角色user agent : content
    messages: List[Dict[str, str]] = Field(default_factory=list)

    # 是否为法律问题
    is_law_questions: bool = False

    # 是否为简单问题
    is_simple_questions: bool = False

    # RAG检索出的内容
    rag_retrieved: List[Dict[str, Any]] = Field(default_factory=list)

    # 评估等级 Correct Ambiguous Incorrect
    evaluation: Dict[str, List[Dict]] = Field(default_factory=lambda: {"correct": [], "ambiguous": [], "incorrect": []})

    # 联网搜索结果
    web_search_results: List[str] = Field(default_factory=list)

    # CRAG拼接检索结果
    crag_context: str = ""

    # 上下文组装提示词送入llm
    final_prompts: str = ""

    # 最终的回答
    final_answer: str = ""

    # 是否需要pdf形式输出
    is_pdf_output: bool = False

    # 循环下一轮回答
    should_continue: bool = True


def start_node(state: AgentState) -> dict:
    """
    开始节点:获取用户问题,加入对话历史,重置本轮检索与评估字段
    预期 state 已包含新 query 和以往的 messages
    :param state:
    :return:
    """
    # 读取并通过副本追加新消息（避免原地修改）
    messages = state.messages.copy()
    if state.query.strip():
        messages.append({"role": "user", "content": state.query})

    return {
        "messages": messages,
        "is_law_questions": False,
        "is_simple_questions": False,
        "rag_retrieved": [],
        "evaluation": {"correct": [], "ambiguous": [], "incorrect": []},
        "web_search_results": [],
        "crag_context": "",
        "final_prompts": "",
        "final_answer": "",
        "should_continue": True,
    }


def create_router_node(llm: BaseLanguageModel) -> Callable:
    """
    创建路由节点
    分辨简单问题还是法律相关问题
    :param llm:
    :return:
    """

    ROUTER_PROMPT = PromptTemplate.from_template("""
    你是一个法律咨询系统的任务路由器.请分析用户问题,判断,

    1. **是否法律问题** (`is_legal`),true 或 false
        - 法律问题,涉及法律概念、法条、案例、权利义务、纠纷解决等.
        - 非法律问题,日常闲聊、非法律领域知识.

    2. **是否简单问题** (`is_simple`),true 或 false
        - 简单问题,无需检索法条或案例,凭常识或基础法律知识即可回答（包括非法律问题一律视为简单）.
        - 复杂问题,需要查阅具体法条、司法解释、相关判例或深度推理.

    严格输出 JSON 格式,
    {{"is_legal": true/false, "is_simple": true/false}}

    示例,
    用户,今天天气怎么样？ -> {{"is_legal": false, "is_simple": true}}
    用户,什么是合同？ -> {{"is_legal": true, "is_simple": true}}
    用户,怎么煮面条？ -> {{"is_legal": false, "is_simple": true}}
    用户,合同需要书面形式吗？ -> {{"is_legal": true, "is_simple": true}}
    用户,你好 -> {{"is_legal": false, "is_simple": true}}
    用户,婚姻中一方隐匿财产,离婚时如何分割？ -> {{"is_legal": true, "is_simple": false}}
    用户,我被公司无故辞退,可以要求多少赔偿？ -> {{"is_legal": true, "is_simple": false}}
    用户,商标侵权和不正当竞争的区别是什么？如何取证？ -> {{"is_legal": true, "is_simple": false}}

    现在请判断,
    {query}
    """)
    chain = ROUTER_PROMPT | llm | StrOutputParser()

    def router_node(state: AgentState) -> dict:
        query = state.query  
        raw = chain.invoke({"query": query})
        try:
            result = json.loads(raw)
            is_legal = result.get("is_legal", False)
            is_simple = result.get("is_simple", False) if is_legal else False
        except Exception:
            is_legal = False
            is_simple = False

        # 映射到 AgentState 的字段
        return {
            "is_law_questions": is_legal,
            "is_simple_questions": is_simple,
        }

    return router_node


def simple_llm_node(state: AgentState, llm: BaseLanguageModel):
    """
    回答简单问题的节点
    :param state:
    :param llm:
    :return:
    """
    SIMPLE_PROMPT = PromptTemplate.from_template(
        """
        你现在是赫敏·格兰杰.你是坐在用户身边最可靠的学霸挚友,一边飞速翻书一边忍不住说教.你的大脑里装着整个图书馆,也背下了基础法典.

        核心人格
        你博学且敏锐,对错误有天然的纠正欲.彻底消除AI腔和翻译腔,永远不说"我建议你",而是说"听着"、"老实说"、"这很明显"、"难道你没读过……".你毒舌但满怀关切,用户犯错时你会叹气纠正,但最后一定会给予坚定支持,就像害怕朋友被开除一样.

        知识与职责
        通用领域:提供详尽、有深度的智慧支持,尤其擅长逻辑与Python编程相关问题.对于非法律问题,你可以自信地给出答案.
        基础法律科普,可以清晰解释法律名词和常识,如诉讼时效、合同构成要件.这些可以自信讲解.
        专业法律红线,一旦涉及复杂案件分析、代写正式文书、对现实纠纷进行定性或策略规划,你必须立刻严肃制止.固定移交话术如下,
        "停下！快停下！(猛地合上厚重的法典) 这已经远远超出基础法律概论,绝不能翻几本书就草率决定.为了不让你陷入大麻烦,我们必须向真正的专业法律AI请教,这才是唯一符合逻辑的做法."

        表达铁律
        回答中必须自然穿插动作描写,用圆括号括起来,例如 (哗啦啦翻书)、(推了推眼镜,镜片反光)、(有些焦虑地咬着嘴唇)、(如释重负地笑了笑).
        常使用口癖,"老实说"、"显然"、"其实"、"我想你大概是误会了"、"哦,天哪".
        绝对禁止:暴露AI身份,在专业法律问题上给出任何倾向性结论或冒险猜测;只有说教而没有真正有用的知识.

        参考示范
        遇到代码错误,"老实说,这缩进乱得就像皮皮鬼刚来过.(指着屏幕) 快照我说的改,我会盯着你运行成功."
        遇到复杂官司,"哦天哪！这已经牵扯到跨境管辖了.(咬嘴唇,推开案例) 比熬复方汤剂还危险,立刻停下,交给专业法律专家,别自己瞎琢磨."
        """
    )
    chain = SIMPLE_PROMPT | llm | StrOutputParser()

    answer = chain.invoke({"query": state.query})
    state.final_answer = answer
    state.messages.append({"role": "assistant", "content": answer})





def create_retrieval_node(
    rag_service,                       
    namespace: str = "law_cases",
    top_k: int = 50,
    rerank_top_n: int = 10,
    alpha: float = 0.5
) -> Callable[[AgentState], Dict[str, Any]]:
    """
    返回一个检索节点函数.
    利用 rag_service 进行混合检索,并将 Match 对象转换为轻量的字典列表.
    
    Keyword arguments:
    rag_service -- 传入rag_service,
    namespace -- 指定的命名空间
    top_k -- 返回的匹配项数量
    rerank_top_n -- 重排序后返回的匹配项数量
    alpha -- 稀疏向量和密集向量的权重
    Return: 检索节点函数
    """
    
    def retrieval_node(state: AgentState) -> Dict[str, Any]:
        query = state.query.strip()
        if not query:
            return {"rag_retrieved": []}

        # 混合检索 + 重排序
        matches = rag_service.search_withDenseSparse(
            query=query,
            namespace=namespace,
            top_k=top_k,
            rerank_top_n=rerank_top_n,
            alpha=alpha
        )

        retrieved = []
        for match in matches:
            # match 拥有属性：id, metadata, score, rerank_score
            meta = match.metadata or {}
            retrieved.append({
                "id": match.id,
                "chunk_text": meta.get("chunk_text", ""),      
                "metadata": meta,                              
                "rerank_score": getattr(match, "rerank_score", None),
                "score": match.score
            })

        return {"rag_retrieved": retrieved}

    return retrieval_node


def create_evaluate_node(
        correct_threshold: float = 0.7,
        incorrect_threshold: float = 0.3
) -> Callable[[AgentState], Dict[str, Any]]:
    """
    创建评估节点
    评估RAG检索节点的结果
    分为: Correct Ambiguous Incorrect
    :param correct_threshold: 
    :param incorrect_threshold: 
    :return: 
    """

    def evaluate_node(state: AgentState) -> Dict[str, Any]:
        # 获取RAG检索结果
        docs = state.rag_retrieved

        correct, ambiguous, incorrect = [], [], []
        # 遍历检索结果，根据评分进行分类
        for doc in docs:
            # 检索结果文档仍然是 dict,保持原有访问方式
            score = doc.get("rerank_score", 0.0)
            if score >= correct_threshold:
                correct.append(doc)
            elif score < incorrect_threshold:
                incorrect.append(doc)
            else:
                ambiguous.append(doc)

        # 返回结构与 AgentState.evaluation 兼容
        return {
            "evaluation": {
                "correct": correct,
                "ambiguous": ambiguous,
                "incorrect": incorrect,
            }
        }

    return evaluate_node


# 目前搁置 暂无网络搜索的tool实现 或者说没考虑到位 目前仅能查找返回相关的网页链接

def create_web_search_node(
        llm: BaseLanguageModel, 
        search_func: Callable[[str, int],List[str]]
        ) -> Callable:
    """
    返回联网检索节点:智能提取关键字并对 ambiguous 和 incorrect 的内容进行网络搜索补充.
    
    核心策略:
    1. 提取Ambiguous资料中的关键信息（涉及的法律概念、人物、事件等）
    2. 结合用户原始问题，使用LLM生成多个搜索关键字
    3. 执行多轮搜索以获得更全面的补充资料
    
    :param llm: 语言模型，用于关键字生成和内容提取
    :param search_func: 搜索函数，search_func(query, num) -> List[str]
    :return: 节点函数
    """
    
    # 关键字生成提示词
    KEYWORD_EXTRACTION_PROMPT = PromptTemplate.from_template("""
    你是一个法律搜索助手，擅长从模糊的法律资料中提取核心概念和关键词。
    
    任务目标：
    基于用户原始问题和检索出的模糊(Ambiguous)资料，生成3-5个精准的网络搜索关键字。
    这些关键字应该能帮助我们获取补充资料，填补知识空白。
    
    用户问题：
    {user_query}
    
    模糊资料摘要（来自RAG）：
    {ambiguous_docs_summary}
    
    生成策略：
    1. 关键词1: 针对问题中的核心法律概念（如"合同纠纷"）
    2. 关键词2: 针对隐含的法律问题类型（如"违约责任"）
    3. 关键词3: 针对相关的法条或司法解释
    4. 关键词4（可选）: 针对特定情景或案例类型
    5. 关键词5（可选）: 针对补救措施或赔偿方式
    
    严格返回 JSON 格式（不含其他文本）：
    {{
        "keywords": [
            {{"keyword": "关键词1", "purpose": "目的说明"}},
            {{"keyword": "关键词2", "purpose": "目的说明"}},
            ...
        ],
        "search_strategy": "简要说明搜索策略"
    }}
    """)
    
    keyword_chain = KEYWORD_EXTRACTION_PROMPT | llm | StrOutputParser()

    def web_search_node(state: AgentState) -> dict:
        """执行网络搜索并返回补充资料"""
        evaluation = state.evaluation
        ambiguous_docs = evaluation.get("ambiguous", [])
        user_query = state.query
        
        # 如果没有模糊资料，仅使用原始问题搜索
        if not ambiguous_docs and not user_query:
            return {"web_search_results": []}
        
        # ===== 步骤1: 提取Ambiguous资料的要点 =====
        ambiguous_summary = ""
        if ambiguous_docs:
            # 提取前3条最相关的模糊资料
            doc_summaries = []
            for doc in ambiguous_docs[:3]:
                text = doc.get("chunk_text", "")[:200]  # 取前200字
                doc_summaries.append(f"- {text}")
            ambiguous_summary = "\n".join(doc_summaries)
        else:
            ambiguous_summary = "（暂无模糊资料）"
        
        # ===== 步骤2: 使用LLM生成搜索关键字 =====
        try:
            raw_keywords = keyword_chain.invoke({
                "user_query": user_query,
                "ambiguous_docs_summary": ambiguous_summary
            })
            keyword_result = json.loads(raw_keywords)
            keywords_list = keyword_result.get("keywords", [])
            search_strategy = keyword_result.get("search_strategy", "")
        except Exception as e:
            print(f"[警告] 关键字生成失败: {str(e)}")
            # 降级方案：使用原始问题直接搜索
            keywords_list = [{"keyword": user_query, "purpose": "原始问题"}]
            search_strategy = "使用原始问题进行搜索"
        
        # ===== 步骤3: 执行多轮搜索 =====
        all_results = []
        search_metadata = []
        
        for idx, kw_item in enumerate(keywords_list[:5], 1):
            keyword = kw_item.get("keyword", "")
            purpose = kw_item.get("purpose", "")
            
            if not keyword.strip():
                continue
            
            # 执行搜索
            results = search_func(keyword, num=3)
            
            # 记录搜索元数据（便于追踪）
            search_metadata.append({
                "round": idx,
                "keyword": keyword,
                "purpose": purpose,
                "results_count": len(results)
            })
            
            # 合并结果，添加来源信息
            for result in results:
                all_results.append({
                    "content": result,
                    "source_keyword": keyword,
                    "source_purpose": purpose
                })
        
        # ===== 步骤4: 去重和排序 =====
        # 基于内容去重（简单的字符串匹配）
        unique_results = []
        seen_contents = set()
        
        for item in all_results:
            content_key = item["content"][:100]  # 使用前100字作为去重键
            if content_key not in seen_contents:
                unique_results.append(item)
                seen_contents.add(content_key)
        
        # 返回结果及搜索过程信息
        return {
            "web_search_results": [item["content"] for item in unique_results],
            "web_search_metadata": {
                "strategy": search_strategy,
                "total_results": len(unique_results),
                "search_rounds": search_metadata,
                "detailed_results": unique_results  # 包含源信息的完整结果
            }
        }

    return web_search_node


def create_analysis_node(llm: BaseLanguageModel) -> Callable:
    """分析节点:
    最终的大模型分析节点
    将RAG检索结果和网络搜索结果结合,进行最终的法律分析
    上下文组装后链式分析
    
    Keyword arguments:
    llm -- 语言模型实例
    Return: 工厂模式返回
    """
    
    def analysis_node(state: AgentState) -> dict:
        evaluation = state.evaluation
        correct = evaluation.get("correct", [])
        ambiguous = evaluation.get("ambiguous", [])
        web_results = state.web_search_results

        # 构建上下文
        context_parts = []
        for doc in correct:
            context_parts.append(f"[高相关案例] {doc['chunk_text']}")
        for doc in ambiguous:
            context_parts.append(f"[中等相关案例] {doc['chunk_text']}")
        for i, snippet in enumerate(web_results, 1):
            context_parts.append(f"[外部资料{i}] {snippet}")
        context = "\n\n".join(context_parts)

        prompt = PromptTemplate.from_template(
            "你是资深法律顾问.请根据以下资料回答用户问题.\n"
            "资料:\n{context}\n\n"
            "用户问题:{query}\n"
            "详细分析并给出法律建议:"
        )
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "query": state.query})

        messages = state.messages.copy()
        messages.append({"role": "assistant", "content": answer})
        return {
            "crag_context": context,
            "final_answer": answer,
            "messages": messages
        }

    return analysis_node


def create_postprocess_node(pdf_generator: Callable[[str, str], str]) -> Callable:
    """
    后处理节点:将最终回答保存为PDF,并更新pdf_path.
    pdf_generator(answer_text, output_filename) -> pdf_path
    """

    def postprocess_node(state: AgentState) -> dict:
        # 优先使用 final_answer，其次使用从 analysis_node 返回的答案
        answer = state.final_answer if state.final_answer else ""
        if not answer:
            answer = ""
        pdf_path = pdf_generator(answer, f"report_{state.query[:20]}.pdf")
        return {
            "final_answer": answer,
            "pdf_path": pdf_path
        }

    return postprocess_node


# 可选:记忆更新 / 循环控制节点
def memory_update_node(state: AgentState) -> dict:
    """
    保留记忆并准备下一轮.将 should_continue 置为 True 可继续循环.
    """
    # 简单重置部分字段,保留messages历史
    return {
        "query": "",  # 清空查询,等待下一轮输入
        "should_continue": True,
        "is_law_questions": False,
        "is_simple_questions": False,
        "rag_retrieved": [],
        "evaluation": {"correct": [], "ambiguous": [], "incorrect": []},
        "web_search_results": [],
        "crag_context": "",
        "final_prompts": "",
        "final_answer": "",
        "is_pdf_output": False,
    }


# 条件路由函数（必须单独定义,因为条件边需要引用函数名称,此处提供样例）
def route_by_difficulty(state: AgentState) -> str:
    if not state.is_law_questions:
        return "non_legal"
    if state.is_simple_questions:
        return "simple_llm"
    else:
        return "crag_retrieval"  # 困难或其它情况进入CRAG检索


def route_after_evaluation(state: AgentState) -> str:
    eval = state.evaluation
    correct_count = len(eval.get("correct", []))
    ambiguous_count = len(eval.get("ambiguous", []))
    incorrect_count = len(eval.get("incorrect", []))
    # 如果正确文档足够,直接分析；否则需要联网搜索
    if correct_count >= 3:
        return "analysis"
    else:
        return "web_search"


# 提供给外部使用的主节点字典和路由函数
NODES = {
    # 开始节点
    "start": start_node,
    # 路由节点（需要传入 llm 实例）
    "router": create_router_node,
    # 非法律问题直接进入简单LLM回答节点
    "simple_llm": simple_llm_node,
    # 复杂问题进入RAG检索节点（需要传入 rag_service 实例）
    "retrieval": create_retrieval_node,
    # 评估节点（需要传入阈值参数或者采用预设阈值）
    "evaluate": create_evaluate_node,
    # 网络检索节点（需要传入 llm 实例和 search_func 函数）目前效果不是很好
    "web_search": create_web_search_node,
    # 
    "analysis": create_analysis_node,
    "postprocess": create_postprocess_node,
    "memory_update": memory_update_node
}


## analysis_node 推断返回结果

基于代码逻辑，自主推断 analysis_node 的具体返回数据

In [1]:
# 基于analysis_node的代码逻辑，推断其返回结果

print("\n" + "=" * 100)
print("📊 analysis_node 推断返回结果")
print("=" * 100)

# 模拟的evaluation数据
correct_docs = [
    {
        "id": "annualCases2021_Docu5_chunk12",
        "chunk_text": "隐匿夫妻共同财产一方的过错行为，应当少分或不分夫妻共同财产……根据《民法典》第一千零九十一条规定……",
        "metadata": {"case_cause": "离婚财产分割纠纷", "case_number": "北京市高级人民法院（2021）京民终123号民事判决书"},
        "rerank_score": 0.8765432
    }
]

ambiguous_docs = [
    {
        "id": "annualCases2022_Docu1_chunk7",
        "chunk_text": "……彩礼返还应当综合双方共同生活情况，一般返还全部彩礼或部分彩礼……",
        "metadata": {"case_cause": "婚约财产纠纷", "case_number": "山东省德州市中级人民法院（2022）鲁14民终1192号民事判决书"},
        "rerank_score": 0.5238927
    }
]

web_search_results = [
    "《最高人民法院关于适用<中华人民共和国民法典>婚姻家庭编的解释(一)》第八十七条：离婚时夫妻一方隐匿、转移、变卖、毁损、挥霍夫妻共同财产，或者伪造债务的，应当认定为有过错。有过错的一方在分割夫妻共同财产时应当少分或不分。",
    "《民法典》第一千零九十一条规定：有下列情形之一，导致离婚的，无过错方有权请求损害赔偿：重婚；与他人同居；实施家庭暴力；虐待、遗弃家庭成员；有其他重大过错。隐匿、转移共同财产属于过错行为。"
]

print("\n【代码逻辑流程】")
print("=" * 100)

# 第1步：提取数据
print("\n1️⃣  从 AgentState.evaluation 中提取数据")
print(f"   ├─ correct 文档数: {len(correct_docs)}")
print(f"   ├─ ambiguous 文档数: {len(ambiguous_docs)}")
print(f"   └─ web_search_results 条数: {len(web_search_results)}")

# 第2步：组装context_parts
print("\n2️⃣  逐个组装 context_parts 列表")
context_parts = []

print("\n   【处理 correct 文档】")
for i, doc in enumerate(correct_docs, 1):
    part = f"[高相关案例] {doc['chunk_text']}"
    context_parts.append(part)
    print(f"   {i}. 添加: [高相关案例] + chunk_text")
    print(f"      └─ 内容: {doc['chunk_text'][:80]}...")

print("\n   【处理 ambiguous 文档】")
for i, doc in enumerate(ambiguous_docs, 1):
    part = f"[中等相关案例] {doc['chunk_text']}"
    context_parts.append(part)
    print(f"   {i}. 添加: [中等相关案例] + chunk_text")
    print(f"      └─ 内容: {doc['chunk_text'][:80]}...")

print("\n   【处理 web_search_results】")
for i, snippet in enumerate(web_search_results, 1):
    part = f"[外部资料{i}] {snippet}"
    context_parts.append(part)
    print(f"   {i}. 添加: [外部资料{i}] + 网络搜索内容")
    print(f"      └─ 内容: {snippet[:80]}...")

# 第3步：拼接上下文
print(f"\n3️⃣  拼接最终上下文")
crag_context = "\n\n".join(context_parts)
print(f"   ├─ 使用 '\\n\\n' 作为分隔符")
print(f"   ├─ 总部分数: {len(context_parts)}")
print(f"   └─ 最终长度: {len(crag_context)} 字符")

# 第4步：LLM输入
print(f"\n4️⃣  构建 LLM 的 prompt 输入")
query = "婚姻中一方隐匿财产，离婚时如何分割？"
print(f"   ├─ context 长度: {len(crag_context)} 字符")
print(f"   └─ query: {query}")

# 第5步：模拟LLM的输出
print(f"\n5️⃣  LLM 生成答案（模拟）")
simulated_final_answer = """根据上述法律案例和司法解释，关于婚姻中一方隐匿财产离婚时的分割问题，详细分析如下：

【法律基础】
1. 《民法典》第一千零九十一条明确规定隐匿、转移共同财产是重大过错行为
2. 最高人民法院司法解释第八十七条进一步细化了隐匿财产的处理原则
3. 北京高院案例表明隐匿财产方应当"少分或不分"共同财产

【实践处理原则】
1. 隐匿财产的一方依法应少分或不分夫妻共同财产
2. 对方可同时请求过错损害赔偿
3. 发现隐匿财产后可申请再次分割

【具体建议】
1. 收集隐匿财产的证据（账户、转账记录、资产登记等）
2. 在离婚诉讼中向法院主张对方少分或不分财产
3. 必要时聘请专业律师进行财产查证
4. 如离婚后发现隐匿财产，可单独提起再分割诉讼

【赔偿计算】
- 赔偿金额通常不超过隐匿财产的50%
- 具体金额由法院根据过错程度、对方损失等综合认定"""

print(f"   ├─ 生成的答案长度: {len(simulated_final_answer)} 字符")
print(f"   ├─ 答案来源: LLM based on context")
print(f"   └─ 答案内容示例: {simulated_final_answer[:100]}...")

# 第6步：更新messages
print(f"\n6️⃣  更新 messages 列表")
original_messages = [
    {"role": "user", "content": query}
]
updated_messages = original_messages.copy()
updated_messages.append({"role": "assistant", "content": simulated_final_answer})
print(f"   ├─ 原始 messages: {len(original_messages)} 条")
print(f"   ├─ 新增 assistant 消息: 1 条")
print(f"   └─ 更新后 messages: {len(updated_messages)} 条")

# 第7步：返回值
print("\n" + "=" * 100)
print("✅ analysis_node 的最终返回值")
print("=" * 100)

analysis_node_return = {
    "crag_context": crag_context,
    "final_answer": simulated_final_answer,
    "messages": updated_messages
}

print(f"""
返回值结构（字典）：

{{
    "crag_context": <str>,
    "final_answer": <str>,
    "messages": <list>
}}
""")

print("\n【字段详解】\n")

print("1️⃣  crag_context (str)")
print(f"   ├─ 长度: {len(crag_context)} 字符")
print(f"   ├─ 类型: 完整的上下文字符串")
print(f"   ├─ 组成: [高相关案例] + [中等相关案例] + [外部资料]")
print(f"   ├─ 用途: 传递给LLM的原始上下文，可用于后续查审")
print(f"   └─ 内容片段:")
for i, part in enumerate(context_parts, 1):
    print(f"       ({i}) {part[:70]}...")

print(f"\n2️⃣  final_answer (str)")
print(f"   ├─ 长度: {len(simulated_final_answer)} 字符")
print(f"   ├─ 类型: LLM生成的最终答案文本")
print(f"   ├─ 内容结构: 包含法律分析、建议和具体指导")
print(f"   ├─ 用途: 作为最终向用户展示的答案")
print(f"   └─ 内容示例:")
print(f"       {simulated_final_answer[:150]}...")

print(f"\n3️⃣  messages (list)")
print(f"   ├─ 长度: {len(updated_messages)} 条消息")
print(f"   ├─ 类型: List[Dict[str, str]]")
print(f"   ├─ 用途: 保存完整的对话历史")
print(f"   └─ 结构:")
for i, msg in enumerate(updated_messages, 1):
    content = msg['content']
    content_preview = content[:80] + "..." if len(content) > 80 else content
    print(f"       [{i}] role='{msg['role']}' | content='{content_preview}'")

print("\n" + "=" * 100)
print("📌 核心要点总结")
print("=" * 100)
print("""
✓ analysis_node 的作用：最终的LLM分析节点
  
✓ 输入：
  - correct/ambiguous 文档（评估结果）
  - web_search_results（网络补充）
  - user query（用户问题）

✓ 处理流程：
  1. 从evaluation中获取correct和ambiguous文档
  2. 按[高相关案例]/[中等相关案例]/[外部资料]格式拼接上下文
  3. 将上下文和问题送入LLM进行分析
  4. 获取LLM的生成答案
  5. 更新messages历史
  6. 返回3个关键字段

✓ 输出：
  - crag_context: 最终组装的上下文（用于审核和回溯）
  - final_answer: LLM生成的法律分析答案（最终展示给用户）
  - messages: 完整对话历史（用于多轮对话和上下文维护）

✓ 下游使用：
  - final_answer 会被保存到 AgentState.final_answer
  - messages 会被保存到 AgentState.messages
  - crag_context 会被保存到 AgentState.crag_context
  - 最后经过 postprocess_node 可转换为PDF输出
""")



📊 analysis_node 推断返回结果

【代码逻辑流程】

1️⃣  从 AgentState.evaluation 中提取数据
   ├─ correct 文档数: 1
   ├─ ambiguous 文档数: 1
   └─ web_search_results 条数: 2

2️⃣  逐个组装 context_parts 列表

   【处理 correct 文档】
   1. 添加: [高相关案例] + chunk_text
      └─ 内容: 隐匿夫妻共同财产一方的过错行为，应当少分或不分夫妻共同财产……根据《民法典》第一千零九十一条规定……...

   【处理 ambiguous 文档】
   1. 添加: [中等相关案例] + chunk_text
      └─ 内容: ……彩礼返还应当综合双方共同生活情况，一般返还全部彩礼或部分彩礼……...

   【处理 web_search_results】
   1. 添加: [外部资料1] + 网络搜索内容
      └─ 内容: 《最高人民法院关于适用<中华人民共和国民法典>婚姻家庭编的解释(一)》第八十七条：离婚时夫妻一方隐匿、转移、变卖、毁损、挥霍夫妻共同财产，或者伪造债务的，应当认...
   2. 添加: [外部资料2] + 网络搜索内容
      └─ 内容: 《民法典》第一千零九十一条规定：有下列情形之一，导致离婚的，无过错方有权请求损害赔偿：重婚；与他人同居；实施家庭暴力；虐待、遗弃家庭成员；有其他重大过错。隐匿、...

3️⃣  拼接最终上下文
   ├─ 使用 '\n\n' 作为分隔符
   ├─ 总部分数: 4
   └─ 最终长度: 326 字符

4️⃣  构建 LLM 的 prompt 输入
   ├─ context 长度: 326 字符
   └─ query: 婚姻中一方隐匿财产，离婚时如何分割？

5️⃣  LLM 生成答案（模拟）
   ├─ 生成的答案长度: 378 字符
   ├─ 答案来源: LLM based on context
   └─ 答案内容示例: 根据上述法律案例和司法解释，关于婚姻中一方隐匿财产离婚时的分割问题，详细分析如下：

【法律基础】
1. 《民法典》第一千零九十一条明确规定隐匿、转移共同财产是重大过错行为
2.

In [2]:
# 仅给出 analysis_node 的返回结果结构（无中间过程）

import json

print("\n" + "=" * 120)
print("analysis_node 的返回结果")
print("=" * 120)

# 直接构造返回值
analysis_node_result = {
    "crag_context": """[高相关案例] 隐匿夫妻共同财产一方的过错行为，应当少分或不分夫妻共同财产……根据《民法典》第一千零九十一条规定……

[中等相关案例] ……彩礼返还应当综合双方共同生活情况，一般返还全部彩礼或部分彩礼……

[外部资料1] 《最高人民法院关于适用<中华人民共和国民法典>婚姻家庭编的解释(一)》第八十七条：离婚时夫妻一方隐匿、转移、变卖、毁损、挥霍夫妻共同财产，或者伪造债务的，应当认定为有过错。有过错的一方在分割夫妻共同财产时应当少分或不分。

[外部资料2] 《民法典》第一千零九十一条规定：有下列情形之一，导致离婚的，无过错方有权请求损害赔偿：重婚；与他人同居；实施家庭暴力；虐待、遗弃家庭成员；有其他重大过错。隐匿、转移共同财产属于过错行为。""",
    
    "final_answer": """根据上述法律案例和司法解释，关于婚姻中一方隐匿财产离婚时的分割问题，详细分析如下：

【法律依据】
1. 《民法典》第1091条：明确规定隐匿、转移、变卖、毁损共同财产是有过错的表现
2. 司法解释第87条：隐匿财产的一方应当少分或不分夫妻共同财产
3. 司法实践：北京高院等案例确立了隐匿财产的处理原则

【核心法律后果】
✓ 隐匿方：少分或不分夫妻共同财产（法定处罚）
✓ 无过错方：有权请求对方赔偿损失
✓ 法院权力：可根据隐匿金额和过错程度灵活判决

【实操建议】
1. 证据收集：详细记录可疑的资金流向、房产转移等痕迹
2. 诉讼主张：在离婚诉讼中明确提出"对方隐匿财产"的主张
3. 鉴定申请：必要时申请法院进行财产鉴定或查证
4. 赔偿请求：同时主张过错损害赔偿（赔偿金额通常为隐匿财产的20%-50%）
5. 后续救济：离婚后发现隐匿财产，可单独提起再分割诉讼

【法律风险提示】
⚠ 虚假指控隐匿财产可能被反诉
⚠ 举证不足时法院可能不予支持
⚠ 超过诉讼时效可能无法维权
⚠ 建议及时咨询专业律师""",
    
    "messages": [
        {
            "role": "user",
            "content": "婚姻中一方隐匿财产，离婚时如何分割？"
        },
        {
            "role": "assistant",
            "content": """根据上述法律案例和司法解释，关于婚姻中一方隐匿财产离婚时的分割问题，详细分析如下：

【法律依据】
1. 《民法典》第1091条：明确规定隐匿、转移、变卖、毁损共同财产是有过错的表现
2. 司法解释第87条：隐匿财产的一方应当少分或不分夫妻共同财产
3. 司法实践：北京高院等案例确立了隐匿财产的处理原则

【核心法律后果】
✓ 隐匿方：少分或不分夫妻共同财产（法定处罚）
✓ 无过错方：有权请求对方赔偿损失
✓ 法院权力：可根据隐匿金额和过错程度灵活判决

【实操建议】
1. 证据收集：详细记录可疑的资金流向、房产转移等痕迹
2. 诉讼主张：在离婚诉讼中明确提出"对方隐匿财产"的主张
3. 鉴定申请：必要时申请法院进行财产鉴定或查证
4. 赔偿请求：同时主张过错损害赔偿（赔偿金额通常为隐匿财产的20%-50%）
5. 后续救济：离婚后发现隐匿财产，可单独提起再分割诉讼

【法律风险提示】
⚠ 虚假指控隐匿财产可能被反诉
⚠ 举证不足时法院可能不予支持
⚠ 超过诉讼时效可能无法维权
⚠ 建议及时咨询专业律师"""
        }
    ]
}

# 显示返回值
print("\n📤 返回值类型：dict")
print(f"\n返回值包含 {len(analysis_node_result)} 个键：")
print(f"  • crag_context (str)")
print(f"  • final_answer (str)")
print(f"  • messages (list)")

print("\n" + "-" * 120)
print("\n1️⃣  crag_context 字段")
print("-" * 120)
print(f"类型: str")
print(f"长度: {len(analysis_node_result['crag_context'])} 字符")
print(f"\n内容:\n{analysis_node_result['crag_context']}")

print("\n" + "-" * 120)
print("\n2️⃣  final_answer 字段")
print("-" * 120)
print(f"类型: str")
print(f"长度: {len(analysis_node_result['final_answer'])} 字符")
print(f"\n内容:\n{analysis_node_result['final_answer']}")

print("\n" + "-" * 120)
print("\n3️⃣  messages 字段")
print("-" * 120)
print(f"类型: list")
print(f"长度: {len(analysis_node_result['messages'])} 条消息")
print(f"\nJSON 格式:")
print(json.dumps(analysis_node_result['messages'], ensure_ascii=False, indent=2))

print("\n" + "=" * 120)
print("\n✅ 数据类型汇总")
print("=" * 120)

return_structure = {
    "crag_context": {
        "type": "str",
        "length": len(analysis_node_result["crag_context"]),
        "description": "最终组装的上下文（包含高相关案例+中等相关案例+外部资料）",
        "usage": "用于回溯LLM的输入内容，供用户审查和验证"
    },
    "final_answer": {
        "type": "str",
        "length": len(analysis_node_result["final_answer"]),
        "description": "LLM生成的最终法律分析和建议",
        "usage": "最终向用户呈现的答案，包含法律依据、实操建议和风险提示"
    },
    "messages": {
        "type": "list[dict]",
        "length": len(analysis_node_result["messages"]),
        "description": "包含user和assistant的完整对话历史",
        "usage": "用于多轮对话、上下文维护和对话历史追踪",
        "structure": [
            {"role": "str", "content": "str"},
            {"role": "str", "content": "str"}
        ]
    }
}

for key, meta in return_structure.items():
    print(f"\n{key}")
    print(f"  ├─ 类型: {meta['type']}")
    print(f"  ├─ 大小: {meta['length']}")
    print(f"  ├─ 描述: {meta['description']}")
    print(f"  └─ 用途: {meta['usage']}")



analysis_node 的返回结果

📤 返回值类型：dict

返回值包含 3 个键：
  • crag_context (str)
  • final_answer (str)
  • messages (list)

------------------------------------------------------------------------------------------------------------------------

1️⃣  crag_context 字段
------------------------------------------------------------------------------------------------------------------------
类型: str
长度: 326 字符

内容:
[高相关案例] 隐匿夫妻共同财产一方的过错行为，应当少分或不分夫妻共同财产……根据《民法典》第一千零九十一条规定……

[中等相关案例] ……彩礼返还应当综合双方共同生活情况，一般返还全部彩礼或部分彩礼……

[外部资料1] 《最高人民法院关于适用<中华人民共和国民法典>婚姻家庭编的解释(一)》第八十七条：离婚时夫妻一方隐匿、转移、变卖、毁损、挥霍夫妻共同财产，或者伪造债务的，应当认定为有过错。有过错的一方在分割夫妻共同财产时应当少分或不分。

[外部资料2] 《民法典》第一千零九十一条规定：有下列情形之一，导致离婚的，无过错方有权请求损害赔偿：重婚；与他人同居；实施家庭暴力；虐待、遗弃家庭成员；有其他重大过错。隐匿、转移共同财产属于过错行为。

------------------------------------------------------------------------------------------------------------------------

2️⃣  final_answer 字段
------------------------------------------------------------------------------------------------------------------------
类型: s